# Prophet SO2 Forecasting Model
## COS40007 Design Project - Smart Government

| Item | Detail |
|---|---|
| Model | Prophet (Meta Forecasting Framework) |
| Target Variable (Y) | air_so2 - Sulfur Dioxide level |
| Predictor Variables (X) | ipi_abs_index_sa (corr=0.68), electricity_local (corr=0.50) |
| Train / Test Split | 40 train months / 20 test months |
| Evaluation Metrics | RMSE, MAE, R-squared |

**Why split=40/20 instead of 48/12?**

The SO2 dataset has only 5 discrete values and the 2022 test period (48/12 split)
has 11 out of 12 values identical at 0.0012, giving near-zero test variance.
With SS_tot near zero, R2 = 1 - SS_res/SS_tot becomes deeply negative even when
predictions are numerically close. The 40/20 split extends the test window back
to May 2021, which includes months where SO2 transitions from 0.001 upward,
giving meaningful variance for a reliable R2 evaluation.

```
ASSIGNMENT/
    data/
        combined_air_electricity_ipi_cleaned.csv
        Prophet.ipynb
        prophet_model.pkl
```

## Section 1 - Library Imports

In [1]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from itertools import product
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from prophet import Prophet

print('All libraries loaded successfully.')
print('If prophet is missing: pip install prophet')

Importing plotly failed. Interactive plots will not work.


All libraries loaded successfully.
If prophet is missing: pip install prophet


## Section 2 - Configuration

In [ ]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
FILE_NAME    = 'combined_air_electricity_ipi_cleaned.csv'

# Resolve the CSV from the project root, with fallbacks for other common layouts.
CSV_CANDIDATES = [
    NOTEBOOK_DIR / FILE_NAME,
    PROJECT_ROOT / 'COS40007DataCleaning' / FILE_NAME,
    PROJECT_ROOT / 'public' / 'data' / FILE_NAME,
    PROJECT_ROOT / 'data' / FILE_NAME,
    PROJECT_ROOT / FILE_NAME,
]
FILE_PATH = next((path for path in CSV_CANDIDATES if path.exists()), CSV_CANDIDATES[0])

# Column names
COL_DATE   = 'date'
COL_TARGET = 'air_so2'
COL_IPI    = 'ipi_abs_index_sa'   # Highest correlation with SO2: 0.68
COL_ELEC   = 'electricity_local'  # Correlation with SO2: 0.50

# Split: 40 train / 20 test
# Test window May 2021 - Dec 2022 includes SO2 transitioning from 0.001 to 0.0013
# giving sufficient variance for a meaningful R2 evaluation.
SPLIT_IDX = 40

# Best hyperparameters found via grid search
BEST_CPS  = 0.08
BEST_SPS  = 0.001
BEST_MODE = 'multiplicative'

if FILE_PATH.exists():
    print(f'CSV found: {FILE_PATH}')
else:
    print('ERROR: CSV not found in any expected location:')
    for candidate in CSV_CANDIDATES:
        print(f'  - {candidate}')

ERROR: CSV not found at d:\Year 3 Sem 1\COS40007 AI Engineering\Assignment\Prophet\Old\combined_air_electricity_ipi_cleaned.csv


## Section 3 - Load Dataset

In [3]:
df_raw = pd.read_csv(FILE_PATH, parse_dates=[COL_DATE])
df_raw = df_raw.sort_values(COL_DATE).reset_index(drop=True)
df = df_raw[[COL_DATE, COL_TARGET, COL_IPI, COL_ELEC]].copy()

print(f'Dataset loaded successfully.')
print(f'  Rows    : {len(df)}')
print(f'  Period  : {df[COL_DATE].min():%b %Y} to {df[COL_DATE].max():%b %Y}')
print(f'  Nulls   : {df.isnull().sum().sum()}')
print()
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'd:\\Year 3 Sem 1\\COS40007 AI Engineering\\Assignment\\Prophet\\Old\\combined_air_electricity_ipi_cleaned.csv'

## Section 4 - Descriptive Statistics and Feature Correlation

In [ ]:
print('=== Descriptive Statistics ===')
print(df[[COL_TARGET, COL_IPI, COL_ELEC]].describe().round(6))
print()
print('=== Correlation with SO2 ===')
corr_cols = ['air_so2','ipi_abs_index_sa','ipi_abs_index',
             'electricity_local','ipi_growth_yoy_index_sa','electricity_total']
print(df_raw[corr_cols].corr()['air_so2'].sort_values(ascending=False).round(4))

## Section 5 - Exploratory Time-Series Plot

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.suptitle('Time-Series Overview: SO2, IPI (SA), Local Electricity (2018-2022)',
             fontsize=12, fontweight='bold')

plot_cfg = [
    (COL_TARGET, '#e74c3c', 'SO2 Level'),
    (COL_IPI,    '#2980b9', 'IPI Absolute Index (Seasonally Adjusted)'),
    (COL_ELEC,   '#27ae60', 'Local Electricity Consumption (GWh)'),
]
for ax, (col, color, label) in zip(axes, plot_cfg):
    ax.plot(df[COL_DATE], df[col], color=color, linewidth=1.8)
    ax.set_ylabel(label, fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('01_time_series_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 01_time_series_overview.png')

## Section 6 - Prepare Data for Prophet
Prophet requires `ds` and `y` columns. `ipi_abs_index_sa` and `electricity_local`
are added as extra regressors using their raw values — Prophet handles
scaling internally when estimating regressor coefficients.

In [ ]:
prophet_df = df.rename(columns={
    COL_DATE  : 'ds',
    COL_TARGET: 'y',
    COL_IPI   : 'ipi_sa',
    COL_ELEC  : 'elec'
})

train_df = prophet_df.iloc[:SPLIT_IDX].copy().reset_index(drop=True)
test_df  = prophet_df.iloc[SPLIT_IDX:].copy().reset_index(drop=True)

print('Train/test split (chronological, no shuffle):')
print(f'  Train: {len(train_df)} months  ({train_df["ds"].min():%b %Y} to {train_df["ds"].max():%b %Y})')
print(f'  Test : {len(test_df)} months   ({test_df["ds"].min():%b %Y} to {test_df["ds"].max():%b %Y})')
print()
print(f'  Test SO2 unique values: {sorted(test_df["y"].unique())}')
print(f'  Test SO2 variance     : {test_df["y"].var():.2e}  (larger = more reliable R2)')

## Section 7 - Walk-Forward Validation (5 Folds)
Walk-forward validation evaluates Prophet stability before final training.
Each fold expands by 3 months to match the small 40-row training set.

In [ ]:
INITIAL_TRAIN = 20
VAL_SIZE      = 3
N_SPLITS      = 5

param_grid = {
    'changepoint_prior_scale': [0.01, 0.05, 0.08, 0.1, 0.3],
    'seasonality_prior_scale': [0.001, 0.01, 0.1, 1],
    'seasonality_mode'       : ['additive', 'multiplicative'],
}

n_combos = (len(param_grid['changepoint_prior_scale']) *
            len(param_grid['seasonality_prior_scale']) *
            len(param_grid['seasonality_mode']))
print(f'Testing {n_combos} combinations x {N_SPLITS} folds...')

best_rmse, best_params, all_results = float('inf'), {}, []

for cps, sps, smode in product(
        param_grid['changepoint_prior_scale'],
        param_grid['seasonality_prior_scale'],
        param_grid['seasonality_mode']):
    fold_rmses = []
    for split in range(N_SPLITS):
        train_end = INITIAL_TRAIN + (split + 1) * VAL_SIZE
        val_end   = train_end + VAL_SIZE
        if val_end > SPLIT_IDX:
            break
        cv_tr = train_df.iloc[:train_end].copy()
        cv_va = train_df.iloc[train_end:val_end].copy()
        try:
            m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                        daily_seasonality=False, changepoint_prior_scale=cps,
                        seasonality_prior_scale=sps, seasonality_mode=smode)
            m.add_regressor('ipi_sa')
            m.add_regressor('elec')
            m.fit(cv_tr)
            fc   = m.predict(cv_va[['ds','ipi_sa','elec']])
            rmse = np.sqrt(mean_squared_error(cv_va['y'].values, fc['yhat'].values))
            fold_rmses.append(rmse)
        except Exception:
            pass
    if fold_rmses:
        avg = np.mean(fold_rmses)
        all_results.append({'cps': cps, 'sps': sps, 'mode': smode, 'rmse': avg})
        if avg < best_rmse:
            best_rmse = avg
            best_params = {'changepoint_prior_scale': cps,
                           'seasonality_prior_scale': sps,
                           'seasonality_mode'       : smode}

print('\nTop 5 combinations by average CV RMSE:')
print(pd.DataFrame(all_results).sort_values('rmse').head(5).to_string(index=False))
print('\nBest parameters:')
for k, v in best_params.items():
    print(f'  {k:<28}: {v}')
print(f'Best CV RMSE: {best_rmse:.6f}')

## Section 8 - Train Final Prophet Model

In [ ]:
model = Prophet(
    changepoint_prior_scale = best_params['changepoint_prior_scale'],
    seasonality_prior_scale = best_params['seasonality_prior_scale'],
    seasonality_mode        = best_params['seasonality_mode'],
    yearly_seasonality      = True,
    weekly_seasonality      = False,
    daily_seasonality       = False,
    interval_width          = 0.95
)
model.add_regressor('ipi_sa')
model.add_regressor('elec')
model.fit(train_df)

print('Prophet model trained successfully.')
for k, v in best_params.items():
    print(f'  {k:<28}: {v}')

## Section 8b - Save Model File
The trained Prophet model is saved as `prophet_model.pkl`.
Reload with `pickle.load(open('prophet_model.pkl', 'rb'))`.

In [ ]:
MODEL_PATH = Path.cwd() / 'prophet_model.pkl'
with open(MODEL_PATH, 'wb') as fh:
    pickle.dump(model, fh)
print(f'Prophet model saved : {MODEL_PATH}')
print(f'File size           : {MODEL_PATH.stat().st_size / 1024:.1f} KB')

## Section 9 - Generate Predictions

In [ ]:
fc_train = model.predict(train_df[['ds','ipi_sa','elec']])
fc_test  = model.predict(test_df[['ds','ipi_sa','elec']])

y_pred_train = fc_train['yhat'].values
y_pred_test  = fc_test['yhat'].values
y_lower_test = fc_test['yhat_lower'].values
y_upper_test = fc_test['yhat_upper'].values
y_actual_train = train_df['y'].values
y_actual_test  = test_df['y'].values

print('Month-by-month test predictions:')
for d, a, p in zip(test_df['ds'], y_actual_test, y_pred_test):
    print(f'  {d.strftime("%b %Y")}: actual={a:.4f}  predicted={p:.6f}  error={a-p:.6f}')

## Section 10 - Evaluate with RMSE, MAE, and R2

In [ ]:
train_rmse = np.sqrt(mean_squared_error(y_actual_train, y_pred_train))
test_rmse  = np.sqrt(mean_squared_error(y_actual_test,  y_pred_test))
train_mae  = mean_absolute_error(y_actual_train, y_pred_train)
test_mae   = mean_absolute_error(y_actual_test,  y_pred_test)
train_r2   = r2_score(y_actual_train, y_pred_train)
test_r2    = r2_score(y_actual_test,  y_pred_test)

print('=' * 50)
print('EVALUATION METRICS')
print('=' * 50)
print(f'\n  TRAIN  RMSE: {train_rmse:.6f}  MAE: {train_mae:.6f}  R2: {train_r2:.4f}')
print(f'  TEST   RMSE: {test_rmse:.6f}  MAE: {test_mae:.6f}  R2: {test_r2:.4f}')
print('=' * 50)
print()
print('Note on split choice:')
print('  The 40/20 split is used because the 20-month test window')
print('  (May 2021 to Dec 2022) includes all 5 distinct SO2 values,')
print('  giving sufficient variance for a reliable R2 evaluation.')
print('  The standard 48/12 split has 11/12 test values = 0.0012')
print('  (near-zero variance), which makes R2 unreliable regardless')
print('  of prediction quality.')

## Section 11 - Actual vs Predicted Visualization

In [ ]:
train_dates = train_df['ds'].values
test_dates  = test_df['ds'].values

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Prophet SO2 Forecast - Actual vs Predicted', fontsize=14, fontweight='bold')

# Training - time series
axes[0,0].plot(train_dates, y_actual_train, label='Actual', lw=2, color='#2c3e50')
axes[0,0].plot(train_dates, y_pred_train, label='Predicted', lw=2, color='#3498db', ls='--')
axes[0,0].set_title('Training Set: Actual vs Predicted', fontweight='bold')
axes[0,0].set_ylabel('SO2 Level')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)
axes[0,0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(axes[0,0].get_xticklabels(), rotation=45)

# Training - scatter
axes[0,1].scatter(y_actual_train, y_pred_train, alpha=0.7, s=40, color='#3498db')
lim = [min(y_actual_train.min(), y_pred_train.min()),
       max(y_actual_train.max(), y_pred_train.max())]
axes[0,1].plot(lim, lim, 'r--', lw=2, label='Perfect Prediction')
axes[0,1].set_title(f'Training Scatter (R2 = {train_r2:.4f})', fontweight='bold')
axes[0,1].set_xlabel('Actual SO2'); axes[0,1].set_ylabel('Predicted SO2')
axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

# Test - time series with 95% CI
axes[1,0].plot(test_dates, y_actual_test, label='Actual', lw=2,
               color='#e74c3c', marker='o', ms=5)
axes[1,0].plot(test_dates, y_pred_test, label='Predicted', lw=2,
               color='#e67e22', ls='--', marker='s', ms=5)
axes[1,0].fill_between(test_dates, y_lower_test, y_upper_test,
                        alpha=0.15, color='#e67e22', label='95% Uncertainty Interval')
axes[1,0].set_title(f'Test Set | RMSE={test_rmse:.6f} | MAE={test_mae:.6f}',
                    fontweight='bold')
axes[1,0].set_ylabel('SO2 Level')
axes[1,0].legend(); axes[1,0].grid(alpha=0.3)
axes[1,0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(axes[1,0].get_xticklabels(), rotation=45)
axes[1,0].text(0.02, 0.97,
    f'RMSE: {test_rmse:.6f}\nMAE: {test_mae:.6f}\nR2: {test_r2:.4f}',
    transform=axes[1,0].transAxes, fontsize=10, va='top',
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))

# Test - scatter
axes[1,1].scatter(y_actual_test, y_pred_test, alpha=0.8, s=50, color='#e67e22')
lim2 = [min(y_actual_test.min(), y_pred_test.min()),
        max(y_actual_test.max(), y_pred_test.max())]
axes[1,1].plot(lim2, lim2, 'r--', lw=2, label='Perfect Prediction')
axes[1,1].set_title(f'Test Scatter (R2 = {test_r2:.4f})', fontweight='bold')
axes[1,1].set_xlabel('Actual SO2'); axes[1,1].set_ylabel('Predicted SO2')
axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('02_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 02_actual_vs_predicted.png')

## Section 12 - Residual Analysis

In [ ]:
residuals = y_actual_test - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Residual Analysis - Test Set', fontsize=13, fontweight='bold')

axes[0].plot(test_dates, residuals, color='#8e44ad', lw=1.5, marker='o', ms=4)
axes[0].axhline(0, color='red', ls='--', lw=1)
axes[0].set_title('Residuals Over Time')
axes[0].set_ylabel('Residual (Actual minus Predicted)')
axes[0].grid(True, alpha=0.3)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(axes[0].get_xticklabels(), rotation=45)

axes[1].hist(residuals, bins=8, color='#8e44ad', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', ls='--', lw=1)
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual Value'); axes[1].set_ylabel('Frequency')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('03_residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 03_residual_analysis.png')
print(f'  Residual mean: {residuals.mean():.6f}  (ideally close to 0)')
print(f'  Residual std : {residuals.std():.6f}')

## Section 13 - Prophet Component Decomposition

In [ ]:
full_df = prophet_df.copy()
full_fc = model.predict(full_df[['ds','ipi_sa','elec']])
fig = model.plot_components(full_fc)
fig.suptitle('Prophet Component Decomposition (Trend, Seasonality, Regressors)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('04_prophet_components.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 04_prophet_components.png')

## Section 14 - Full Forecast Plot

In [ ]:
fig = model.plot(full_fc)
ax  = fig.gca()
ax.axvline(x=test_df['ds'].iloc[0], color='red', ls='--', lw=1.5,
           label='Train/Test Split')
ax.legend(fontsize=9)
ax.set_title('Prophet Full Forecast - SO2 with 95% Uncertainty Interval',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('SO2 Level')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('05_prophet_full_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 05_prophet_full_forecast.png')

## Section 15 - Save Results Summary

In [ ]:
summary = pd.DataFrame({
    'Model'      : ['Prophet'],
    'Split'      : [f'{SPLIT_IDX} train / {60-SPLIT_IDX} test'],
    'Train RMSE' : [round(train_rmse, 6)],
    'Test RMSE'  : [round(test_rmse,  6)],
    'Train MAE'  : [round(train_mae,  6)],
    'Test MAE'   : [round(test_mae,   6)],
    'Train R2'   : [round(train_r2,   4)],
    'Test R2'    : [round(test_r2,    4)],
})
summary.to_csv('prophet_results_summary.csv', index=False)
print('Saved: prophet_results_summary.csv')
print()
print(summary.to_string(index=False))
print(f'\nProphet SO2 forecast completed successfully.')